<a href="https://colab.research.google.com/github/SY-256/llms-from-scratch/blob/main/notebooks/dpo-from-scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLMアライメントのための直接選好最適化（DPO）の実装

## DPO
- ユーザーの期待や指示に合ったレスポンスを生成するようにモデルをファインチューニング（アライメント）する手法
- 一方の回答スタイルを他方より好むようにLLMを教育する手法。ユーザーの選好にそって調整する
- 人間の選好または特定の目標に合わせてモデルの出力を直接最適化することに焦点を当てている

## 2.1 データセットの読み込み

In [ ]:
import json
import os
import requests

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    data = json.loads(text_data)
    return data


file_path = "instruction-data-with-preference.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/04_preference-tuning-with-dpo/instruction-data-with-preference.json"
)

data = download_and_load_file(file_path, url)
print(f"Number of entries: {len(data)}")

In [ ]:
# 2つのサンプルエントリを確認
import pprint

pprint.pp(data[50])

In [ ]:
pprint.pp(data[999])

`'chosen'`と`'rejected'`エントリはDPOに使用するもので、`'chose'`が選好された応答、`'rejected'`が非選好の応答

- ファインチューニングの目標は、モデルが拒否された応答ではなく、選好された応答のスタイルに従うようにすること

In [ ]:
# Alpacaプロンプトスタイルを適用してモデル入力をフォーマットするユーティリティ関数
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else " "

    return instruction_text + input_text

In [ ]:
# サンプルでフォーマット適用確認
model_input = format_input(data[50])
print(model_input)

In [ ]:
# Alpacaプロンプトスタイルを使用して、選好された応答と拒否された応答をフォーマットする
desired_response = f"### Response:\n{data[50]['chosen']}"
print(desired_response)

In [ ]:
possible_response = f"### Response:\n{data[50]['rejected']}"
print(possible_response)

## 2.2 学習・検証・テストデータの分割
- 学習データ:85%, 検証データ:5%、テストデータ:10%に分割

In [ ]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
val_portion = int(len(data) - train_portion - test_portion)

train_data = data[:train_portion]
val_data = data[train_portion+test_portion:]
test_data = data[train_portion:train_portion + test_portion]

In [ ]:
print(f"Training set length: {len(train_data)}")
print(f"Validation set length: {len(val_data)}")
print(f"Test set length: {len(test_data)}")

## 2.3 `PreferenceDataset`クラスとバッチ処理関数の開発
- 単一の出力シーケンス（応答）に注目する代わりに、一方が他方よりも選好（「chosen」）される応答ペアを返すようにデータセットクラスを修正する

In [ ]:
import torch
from torch.utils.data import Dataset

class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # テキストを事前にトークン化
        self.encoded_texts = []
        for entry in data:
            prompt = format_input(entry)
            rejected_response = entry["rejected"]
            chosen_response = entry["chosen"]

            prompt_tokens = tokenizer.encode(prompt)
            chosen_full_text = f"{prompt}\n\n### Response:\n{chosen_response}"
            rejected_full_text = f"{prompt}\n\n### Response:\n{rejected_response}"
            chosen_full_tokens = tokenizer.encode(chosen_full_text)
            rejected_full_tokens = tokenizer.encode(rejected_full_text)

            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": chosen_full_tokens,
                "rejected": rejected_full_tokens,
            })

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

- 更新された`PreferenceDataset`クラスとともに、バッチ内の各シーケンスを同じ長さにパディングするために使用するバッチレコード関数も更新する必要がある。これにより、バッチとしてまとめることができる

In [ ]:
def custom_collate_fn(
        batch,
        pad_token_id=50256,
        allowed_max_length=None,
        mask_prompt_tokens=True,
        device="cpu"
):
    # バッチデータを保持するリストを初期化
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [],
        "chosen_mask": [],
    }

    # 共通パディング長を設定するために最も長いシーケンスを確定
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max(len(item[key])+1 for item in batch)
            max_length_common = max(max_length_common, current_max)

    # バッチ内の各アイテムを処理
    for item in batch:
        prompt = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt)

        for key in ["chosen", "rejected"]:
            # 共通最大長に応じてパディングを調整
            sequence = item[key]
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))
            mask = torch.ones(len(padded)).bool()

            # すべてのパディングトークンのマスクをFalseに設定
            mask[len(sequence):] = False

            # すべての入力トークンのマスクをFalseに設定
            # +2 は"### Response" の前の2つ改行("\n")トークンをFalseに設定
            if mask_prompt_tokens:
                mask[:prompt.shape[0]+2] = False

            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)

    # 最終処理
    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        # 指定されたキーのすべてのシーケンスをテンソルにスタック
        tensor_stack = torch.stack(batch_data[key])

        # オプションで最大シーケンス長に切り詰め
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]

        # 指定されたデバイスに移動
        batch_data[key] = tensor_stack.to(device)

    return batch_data

In [ ]:
# サンプルで確認
from functools import partial

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    mask_prompt_tokens=True,
    allowed_max_length=1024 # モデルがサポートするコンテキストサイズの最大を設定
)

In [ ]:
# サンプルで確認
example_data = data[:2]

for i in example_data:
    print()
    pprint.pp(i)

In [ ]:
import tiktoken
from torch.utils.data import DataLoader

tokenizer = tiktoken.get_encoding("gpt2")

example_dataset = PreferenceDataset(example_data, tokenizer)

example_dataloader = DataLoader(
    example_dataset,
    batch_size=2,
    collate_fn=customized_collate_fn,
    shuffle=False
)

In [ ]:
# データセットのキーを確認
for batch in example_dataloader:
    break

print(f"batch.keys: {batch.keys()}")

In [ ]:
# promptはテンソルのリストで、各テンソルには指定されたサンプルのトークンIDが含まれている
# バッチサイズは2を指定したので、2つのトークンIDテンソルのリストがある
batch["prompt"]

In [ ]:
# chosenとrejectedの応答エントリはテンソルとしてスタックできるようにパディングされている
batch["chosen"]

In [ ]:
# トークンIDからテキストに戻すためのユーティリティ関数
def decode_tokens_from_batch(token_ids, tokenizer):
    ids_in_python_list = token_ids.flatten().tolist()
    return tokenizer.decode(ids_in_python_list)

In [ ]:
# バッチ内の最初のプロンプトエントリに`decode_tokens_from_batch`ユーティリティ関数を適用
text = decode_tokens_from_batch(
    token_ids=batch["prompt"][0],
    tokenizer=tokenizer
)
print(text)

In [ ]:
# `chosen`に対しても同じように変換してみる
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0],
    tokenizer=tokenizer
)
print(text)

- `<|endoftext|>`のパディングトークンは損失計算で無視されるため、学習結果に影響しない

In [ ]:
# 同じように`rejected`についても確認
text = decode_tokens_from_batch(
    token_ids=batch["rejected"][0],
    tokenizer=tokenizer
)
print(text)

データマスクについて:
- 各データセットエントリに対して`"chosen_mask"`と`"rejected_mask"`を作成している
- マスクは`"chosen"`エントリに示すように、応答エントリと同じ形状を持つ：

In [ ]:
print(f"chosen inputs: {batch['chosen'][0].shape}")
print(f"chosen mask: {batch['chosen_mask'][0].shape}")

In [ ]:
# マスクエントリの内容は真理値(bool型)
batch["chosen_mask"][0]

- `True`値は実際の応答に対応するトークンIDを返す
- `False`トークンは、プロンプトトークン（`customized_collate_fn`関数で`mask_prompt_tokens=True`を設定した場合）またはパディングトークンに対応するトークンIDを示す
- マスクを選択マスクとして使用して、応答に対する（`True`の）トークンIDのみを選択できる。つまり、すべてのプロンプトとパディングトークンを取り除くことができる

In [ ]:
# 取り除けるか確認
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0][batch["chosen_mask"][0]], # TrueのものだけトークンIDが返ってくる
    tokenizer=tokenizer
)
print(text)

In [ ]:
text = decode_tokens_from_batch(
    token_ids=batch["rejected"][0][batch["rejected_mask"][0]],
    tokenizer=tokenizer
)
print(text)

- このマスクを使用して、後でDPO損失を計算する際にプロンプトとパディングトークンを無視する

## 2.4 学習・検証・テストセットのデータローダー作成

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = PreferenceDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [ ]:
val_dataset = PreferenceDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

test_dataset = PreferenceDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

In [ ]:
# データローダーからデータセットの形状を確認
for batch in train_loader:
    print(
        batch["chosen"].shape,
        batch["rejected"].shape
    )

- 各行は各バッチの`chosen`と`rejected`エントリの形状を示している
- バッチごとにパディングを適用しているため、各行の形状が異なる
- 効率化のためであり、データセット全体で最長のサンプルにすべてのサンプルをパディングする（全体で同じ長さになる）のは非効率であるから

In [ ]:
# Google Driveのモデルパス
model_path = "/content/drive/MyDrive/colab-notebooks/LLMs-from-scratch/model/gpt2-medium355M-sft.pth"